In [1]:
# build genAI application with RAG pieline system(astradb)
# setup env variables
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
# load_documents
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("./ai_papers")
pdf_documents = loader.load()

print(len(pdf_documents))

C:\Users\mriyu\AppData\Local\Temp\ipykernel_23468\1332824094.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\mriyu\miniconda3\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


11


In [3]:
# split into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size =1000, chunk_overlap=200)
documents = splitter.split_documents(pdf_documents)
print(len(documents)) 
documents[:5]

43


[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [ ]:
# embedding and vector store

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_astradb import AstraDBVectorStore

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = AstraDBVectorStore(
    collection_name="ragstck5oct2026_v2",
    embedding=embedding,
    api_endpoint="https://777b60f7-b672-4f51-be31-e929b4f7d9de-us-east-2.apps.astra.datastax.com",
    token="AstraCS:xxxx:xxxx",
)

vector_store

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3021.45it/s]


In [6]:
vector_store.add_documents(documents)

['f68e1c2dfb3a438f9b66818b23353579',
 '1230f6e9a34940e28cbbca5a5df9d98a',
 'c511e3195bd049d384ccf28cedd89996',
 '42f580e81e2942638fc27922580c2015',
 'f7282b559cca4dc88973c971f9293de4',
 '6d67be6eceae41ea8db13f2ec74d25b7',
 'd057a49b37f44477852f1ba189567b5f',
 '7b317f7817434a09a66d79fd9bfd3f83',
 'bd75bfbd72264eb1bfa67b1c89425884',
 '96025a2c1ed84761800a74af7c1deb16',
 'de2e3d18ee7a4ef1864e6ebadf99d75d',
 '9bc6371f89634362a37ac02755137489',
 'ca7234d6fbda4d2589ca34261d240f5b',
 '8b0dd34cabe24472afd74774b2995c2f',
 'd73d11aed15b4eb68f5a2d3c684d0880',
 'a8f449f2cd2c4e239ea97b32ac937f01',
 'a14b67d5261744408d931fb17c4eb2e8',
 '72a1d1a4dabf40bea2a60ad13db6d6c8',
 '2e9f03b91dfa45688b3f63f276341aca',
 '8c506e41da404714a2f8e9915a0e7b42',
 '4d374bba8a7d4ca2b1de58f9502e6cb1',
 'dbfc9dacd4b243f889657ea9a338e841',
 '564fca3d3b804f15a923193c5ee5d44a',
 '1e91229e4ce748679efbf066da4a61c6',
 'a2f8f2cad4f74a3cba73617e078419be',
 '7380090dd3b74364946c6a5f5fb03bfe',
 '8aa1b5c8c1d24f61a394e237fb7e2a39',
 

In [7]:
import langchain
import langchain_community
import astrapy

print("langchain:", langchain.__version__)
print("langchain-community:", langchain_community.__version__)
print("astrapy:", astrapy.__version__)

langchain: 1.3.15
langchain-community: 0.4.2
astrapy: 2.3.1


In [8]:
retriever = vector_store.as_retriever()

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
# llm
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)
#prompt
prompt = ChatPromptTemplate.from_template(
    """
You are helpfuyl AI assistant . PLease respond to the user's Question
  question:
  {input}
  context:
  {context}
    """
    
)

In [10]:
# chains
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
doc_chain = create_stuff_documents_chain(llm, prompt)
doc_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\nYou are helpfuyl AI assistant . PLease respond to the user's Question\n  question:\n  {input}\n  context:\n  {context}\n    "), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'vi

In [11]:
# retriever chain
from langchain_classic.chains import create_retrieval_chain
retriever_chain = create_retrieval_chain(retriever, doc_chain)

In [12]:
# generate response
response = retriever_chain.invoke({"input":"What is transformer?"})
response

{'input': 'What is transformer?',
 'context': [Document(id='de2e3d18ee7a4ef1864e6ebadf99d75d', metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-Ger